In [2]:
#Build similarity matrix
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import pandas as pd

df=pd.read_csv(r"C:\Users\Admin\Desktop\My stuff\Programming\ML\Projects\spotify-recommendation-system\data\processed\tracks_with_moods.csv")
feature_cols=['danceability','energy','valence','acousticness','tempo','loudness','speechiness','instrumentalness']
X=df[feature_cols].values

In [4]:
#On demand recommender function
def get_recommendations(track_name, df, feature_cols, n=10):

    #Find the song
    matches=df[df['track_name'].str.lower()==track_name.lower()]
    if matches.empty:
        print(f"Song '{track_name}' not found!")
        return None

    song = matches.iloc[0]
    song_idx = song.name
    
    # Get its feature vector
    song_vector = df.loc[song_idx, feature_cols].values.reshape(1, -1)
    
    # Compute similarity against all songs
    all_vectors = df[feature_cols].values
    similarities = cosine_similarity(song_vector, all_vectors)[0]
    
    # Sort by similarity, skip the song itself (index 0)
    similar_indices = similarities.argsort()[::-1][1:n+1]
    
    results = df.iloc[similar_indices][['track_name', 'artists', 'track_genre', 'mood', 'popularity']]
    results['similarity_score'] = similarities[similar_indices].round(4)
    
    return results
# Test it
recs = get_recommendations("Shape of You", df, feature_cols)
print(recs)
        

                                              track_name  \
64100  Les Indes Galantes - Air pour les esclaves afr...   
47690                                       Sink Lateral   
6798                                           Reno Ride   
54870                                      Inner Courage   
46285                                   Dill Pickles Rag   
40744                                     Keali'i's Mele   
46184                             Yes Sir That's My Baby   
46677                            Darktown Strutters Ball   
40558                                      Minha Saudade   
85779                                      Milonga Brava   

                                                 artists track_genre   mood  \
64100  Jean-Philippe Rameau;Jordi Savall;Le Concert D...       opera  Chill   
47690                                              Ilkae         idm  Focus   
6798                                      Clarence White   bluegrass  Focus   
54870                  

In [7]:
#Mood based filter
def get_mood_playlist(mood, df, n=10):
    """
    Returns top n most popular songs in a given mood cluster.
    """
    mood_tracks = df[df['mood'] == mood].sort_values('popularity', ascending=False)
    return mood_tracks[['track_name', 'artists', 'popularity']].head(n)

# Test it
print(get_mood_playlist('Party', df))


                                              track_name  \
45781              Quevedo: Bzrp Music Sessions, Vol. 52   
66009                                    I Ain't Worried   
67636                                          As It Was   
19721                                            CUFF IT   
19299                                  Super Freaky Girl   
67700                                          As It Was   
57495                                             LOKERA   
19097  Vegas (From the Original Motion Picture Soundt...   
19682                                            Ferrari   
57720                                       La Corriente   

                           artists  popularity  
45781             Bizarrap;Quevedo        0.99  
66009                  OneRepublic        0.96  
67636                 Harry Styles        0.95  
19721                      Beyoncé        0.93  
19299                  Nicki Minaj        0.92  
67700                 Harry Styles        0.9

COLLABORATIVE FILTERING 

In [8]:
#Simulate user-item matrix
# Concept: popularity score = proxy for "many users played this"
# We'll create 1000 synthetic users who each "listen to" tracks from a few genres

import numpy as np

np.random.seed(42)
genres = df['track_genre'].unique()
n_users = 1000
n_tracks = len(df)

# Create a sparse user-track interaction matrix
# Each user has affinity for 3 random genres
user_item = np.zeros((n_users, n_tracks))
for user in range(n_users):
    fav_genres = np.random.choice(genres, 3, replace=False)
    genre_mask = df['track_genre'].isin(fav_genres)
    play_probs = (df['popularity'] / 100) * genre_mask.astype(float)
    played = np.random.rand(n_tracks) < play_probs.values
    user_item[user] = played.astype(float)


In [9]:
from sklearn.decomposition import TruncatedSVD
from scipy.sparse import csr_matrix

# Convert to sparse (efficient for memory)
user_item_sparse = csr_matrix(user_item)

# SVD reduces the matrix into latent factors
svd = TruncatedSVD(n_components=50, random_state=42)
user_factors = svd.fit_transform(user_item_sparse)
item_factors = svd.components_.T  # shape: (n_tracks, 50)

# Recommend for a user: find tracks most aligned with their latent vector
def collab_recommend(user_id, user_factors, item_factors, df, n=10):
    user_vec = user_factors[user_id]
    scores = item_factors @ user_vec
    top_indices = scores.argsort()[::-1][:n]
    return df.iloc[top_indices][['track_name', 'artists', 'track_genre', 'popularity']]

print(collab_recommend(42, user_factors, item_factors, df))


                                        track_name  \
10738                              I'll Be Waiting   
44998                                Azrael - Live   
10760                               Under the Tree   
77811                 Desert Moon - From "Aladdin"   
81510                                   Creo en Ti   
6497                   When You Say Nothing At All   
45137                               Triste Funeral   
10815                            Evacuating London   
44834  La Frontera Inesperada - Remasterizado 2016   
45151                                  Franz Weiss   

                             artists  track_genre  popularity  
10738                          Adele      british        0.54  
44998                         U.D.O.  heavy-metal        0.20  
10760                   Sam Palladio      british        0.40  
77811       Mena Massoud;Naomi Scott   show-tunes        0.48  
81510                    Miguel Bosé      spanish        0.36  
6497   Alison Krauss 